# 04 — Modelo fundacional Chronos-2 (Amazon)

**RESPIR-AI · TFG** — Predicción horaria de la OUR con el modelo fundacional de series temporales
**amazon/chronos-2** (119.5 M de parámetros, `chronos-forecasting` 2.3.1, contexto máximo 8192,
usado aquí con **C = 512 h** según el protocolo v2).

Seis escenarios sobre las mismas 36 ventanas rolling-origin del protocolo:

- **E1 — ZS univariante**: zero-shot, solo el histórico de OUR.
- **E2 — ZS multivariante**: zero-shot + 13 covariables de proceso como *past covariates*.
- **E3 — ZS multi + futuras**: E2 + 6 covariables meteorológicas como *known future covariates* (pronóstico real).
- **E4 — FT univariante**: fine-tuning LoRA (500 pasos, lr=1e-5, contexto 512) sobre train, validación para selección de checkpoint.
- **E5 — FT multi**: fine-tuning LoRA con covariables de proceso pasadas.
- **E6 — FT multi + futuras**: fine-tuning LoRA con meteo declaradas como conocidas a futuro (`known_covariates_names`).

**Nota de viabilidad en 8 GB**: la API pública `Chronos2Pipeline.fit()` soporta `finetune_mode="lora"`.
Con contexto 512, `batch_size=32` y 500 pasos, la memoria pico fue ≈2.0–2.3 GB, muy dentro de los
8 GB de la RTX 3070; el fine-tuning completo (`full`) también sería viable pero LoRA es la opción
recomendada para adaptación con pocos datos. Cada fine-tuning tardó ≈51–55 s. Semilla 42.

## 1. Carga del modelo y de la infraestructura común

Se reutiliza `common_eval.py` (mismo módulo que el cuaderno 03).

In [ ]:
import warnings, time, os
import numpy as np, pandas as pd, torch
from chronos import Chronos2Pipeline
from chronos.chronos2 import preprocess as c2pre
from common_eval import *
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

df, proto = load_all()
origins = get_origins(df, proto)
Y = true_targets(df, origins)

pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
n_params_c2 = sum(p.numel() for p in pipe.model.parameters())
print(f"Parámetros: {n_params_c2/1e6:.1f} M · contexto máx del modelo: {pipe.model_context_length}")

## 2. Construcción de entradas por ventana

Para cada origen t0: target = 512 h de OUR hasta t0; covariables de proceso como `past_covariates`; meteo como `future_covariates` (48 h reales, equivalentes a pronóstico) cuando el escenario lo pide.

In [ ]:
def build_inputs(with_proc=False, with_fut_meteo=False):
    dicts = []
    for o in origins:
        i = o['iloc']
        d = {"target": df['our'].iloc[i-511:i+1].to_numpy(dtype=np.float32)}
        past = {}
        if with_proc:
            for c in PROC:
                past[c] = df[c].iloc[i-511:i+1].to_numpy(dtype=np.float32)
        if with_fut_meteo:
            for c in METEO:
                past[c] = df[c].iloc[i-511:i+1].to_numpy(dtype=np.float32)
            d["future_covariates"] = {c: df[c].iloc[i+1:i+49].to_numpy(dtype=np.float32)
                                      for c in METEO}
        if past:
            d["past_covariates"] = past
        dicts.append(d)
    return dicts

def run_chronos(pipeline, escenario, with_proc, with_fut, desc):
    """Una única predicción de 48 pasos por origen (mediana -> media), recortada a cada H."""
    dicts = build_inputs(with_proc, with_fut)
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()
    qs, means = pipeline.predict_quantiles(dicts, prediction_length=48)
    torch.cuda.synchronize()
    t_inf = (time.perf_counter() - t0) / len(origins)
    mem = torch.cuda.max_memory_allocated() / 1e6
    P = np.stack([m.squeeze().numpy() for m in means]).astype(float)
    rows, wins = evaluate_preds(P, Y, "chronos2", escenario, t_inf, 512, origins,
                                extra={"covariables": desc})
    print(f"{escenario}: MAE@6={rows[0]['MAE']:.3f} MAE@48={rows[-1]['MAE']:.3f} "
          f"| {t_inf*1000:.0f} ms/ventana | {mem:.0f} MB")
    return P, rows, wins, t_inf, mem

## 3. Escenarios zero-shot (E1–E3)

In [ ]:
c2_rows, c2_win, c2_cost = [], [], []
for esc, wp, wf, desc in [
        ("E1_zs_univariante",  False, False, "ninguna"),
        ("E2_zs_multi_pasadas", True, False, "proceso pasadas"),
        ("E3_zs_multi_futuras", True, True,  "proceso pasadas + meteo futuras reales")]:
    P, rows, wins, t_inf, mem = run_chronos(pipe, esc, wp, wf, desc)
    c2_rows.extend(rows); c2_win.extend(wins)
    c2_cost.append({"modelo": "chronos2", "escenario": esc, "n_parametros": n_params_c2,
                    "t_train_s": 0.0, "t_inf_ventana_gpu_s": t_inf,
                    "t_inf_ventana_cpu_s": np.nan, "mem_gpu_pico_MB": mem,
                    "dispositivo": "GPU"})

## 4. Fine-tuning LoRA (E4–E6)

Series de entrenamiento: los 12 segmentos operativos del bloque de train (491–2171 h cada uno);
validación: los 2 trozos del bloque de val (segmentos 11 y 12). Se ajusta con
`finetune_mode="lora"`, `learning_rate=1e-5` (valor recomendado por la librería para LoRA),
`num_steps=500`, `context_length=512`, `batch_size=32`, con selección del mejor checkpoint por
pérdida de validación. En E6 las covariables meteorológicas se declaran conocidas a futuro con
`known_covariates_names` a través de `chronos.chronos2.preprocess.from_list_of_dicts`.

In [ ]:
train_end_ts = pd.Timestamp(proto['particion']['train']['fin'])
val_start_ts = pd.Timestamp(proto['particion']['val']['inicio'])
val_end_ts   = pd.Timestamp(proto['particion']['val']['fin'])

def contiguous_series(mask):
    sub = df[mask & (df.segment_id >= 0)]
    return [g for _, g in sub.groupby('segment_id')]

tr_chunks = contiguous_series(df.index <= train_end_ts)
va_chunks = contiguous_series((df.index >= val_start_ts) & (df.index <= val_end_ts))

def ft_inputs(chunks, with_proc=False, with_meteo_known=False):
    dicts = []
    for g in chunks:
        if len(g) < 96: continue
        d = {"target": g['our'].to_numpy(dtype=np.float32)}
        past = {}
        if with_proc:
            for c in PROC: past[c] = g[c].to_numpy(dtype=np.float32)
        if with_meteo_known:
            for c in METEO: past[c] = g[c].to_numpy(dtype=np.float32)
        if past: d["past_covariates"] = past
        dicts.append(d)
    return dicts

def finetune_and_eval(esc, with_proc, with_known_meteo, desc):
    if with_known_meteo:
        tr = c2pre.from_list_of_dicts(ft_inputs(tr_chunks, with_proc, True),
                                      prediction_length=48, known_covariates_names=METEO)
        va = c2pre.from_list_of_dicts(ft_inputs(va_chunks, with_proc, True),
                                      prediction_length=48, known_covariates_names=METEO)
    else:
        tr = ft_inputs(tr_chunks, with_proc)
        va = ft_inputs(va_chunks, with_proc)
    torch.manual_seed(42); np.random.seed(42)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    pipe_ft = pipe.fit(tr, prediction_length=48, validation_inputs=va,
                       finetune_mode="lora", learning_rate=1e-5, num_steps=500,
                       context_length=512, batch_size=32,
                       output_dir=f"c2_{esc}", remove_printer_callback=True,
                       logging_strategy="no")
    t_ft = time.perf_counter() - t0
    mem_ft = torch.cuda.max_memory_allocated() / 1e6
    P, rows, wins, t_inf, mem = run_chronos(pipe_ft, esc, with_proc, with_known_meteo, desc)
    c2_rows.extend(rows); c2_win.extend(wins)
    c2_cost.append({"modelo": "chronos2", "escenario": esc, "n_parametros": n_params_c2,
                    "t_train_s": t_ft, "t_inf_ventana_gpu_s": t_inf,
                    "t_inf_ventana_cpu_s": np.nan,
                    "mem_gpu_pico_MB": max(mem, mem_ft), "dispositivo": "GPU"})
    del pipe_ft; torch.cuda.empty_cache()

finetune_and_eval("E4_ft_univariante",   False, False, "ninguna (LoRA 500 pasos)")
finetune_and_eval("E5_ft_multi_pasadas", True,  False, "proceso pasadas (LoRA 500 pasos)")
finetune_and_eval("E6_ft_multi_futuras", True,  True,  "proceso pasadas + meteo futuras (LoRA 500 pasos)")

pd.DataFrame(c2_rows).to_csv("results/resultados_chronos2.csv", index=False)

## 5. Resultados obtenidos (ejecución real) — MAE por escenario y horizonte

| Escenario | H=6 | H=12 | H=24 | H=48 |
|---|---|---|---|---|
| E1_zs_univariante | 3.631 | 4.009 | 4.119 | 4.535 |
| E2_zs_multi_pasadas | 3.672 | 4.025 | 4.051 | 4.407 |
| E3_zs_multi_futuras | 3.668 | 3.978 | 3.976 | 4.198 |
| E4_ft_univariante | 3.658 | 3.949 | 4.027 | 4.474 |
| E5_ft_multi_pasadas | 3.661 | 4.021 | 4.049 | 4.426 |
| E6_ft_multi_futuras | 3.668 | 3.992 | 4.001 | 4.230 |

**Lectura**: en zero-shot univariante Chronos-2 queda ligeramente por detrás del baseline XGBoost
en horizontes cortos (3.63 vs 3.43 a H=6) y similar a H=48. Las covariables de proceso pasadas (E2)
apenas aportan, pero las **meteo futuras reales (E3) sí mejoran claramente el horizonte largo**:
MAE@48 pasa de 4.535 (E1) a 4.198 (E3), la mejor configuración de toda la familia. El fine-tuning
LoRA (E4–E6) produce mejoras pequeñas respecto a sus equivalentes zero-shot en horizontes medios
(E4 mejora E1 a H=12–24) pero no supera al zero-shot con futuras (E3): con ~14 000 h de datos de
una sola planta, el conocimiento previo del modelo fundacional domina sobre la adaptación local.
Coste: fine-tuning ≈51–55 s por escenario y ≤2.3 GB de VRAM; inferencia 2–28 ms por ventana en GPU.

## Resultados guardados

Este cuaderno requiere horas de GPU para re-ejecutarse por completo; la celda siguiente carga los resultados ya calculados desde `results/` y puede ejecutarse en cualquier máquina.

In [1]:
# Resultados guardados (esta celda se puede ejecutar sin GPU)
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"common_eval.py").exists() else Path.cwd().parent
os.chdir(ROOT)
import pandas as pd
d = pd.read_csv("results/resultados_chronos2.csv")
print(d.pivot(index="escenario", columns="H", values="MAE").round(3).to_string())

H                       6      12     24     48
escenario                                      
E1_zs_univariante    3.631  4.009  4.119  4.535
E2_zs_multi_pasadas  3.672  4.025  4.051  4.407
E3_zs_multi_futuras  3.668  3.978  3.976  4.198
E4_ft_univariante    3.658  3.949  4.027  4.474
E5_ft_multi_pasadas  3.661  4.021  4.049  4.426
E6_ft_multi_futuras  3.668  3.992  4.001  4.230
